# CG Water O-O RDF Validation

This notebook validates fastMD's CG water model by comparing the
O-O radial distribution function g(r) against a LAMMPS reference.

**Prerequisite:** `dump_freq 150` set in `fastmd_nvt.conf` before simulation.
The simulation produces `traj.bin` in this directory.


In [ ]:
import struct
import os
import numpy as np

def read_trajectory(path):
    """Read fastMD binary trajectory. Returns dict with natoms, ntypes, frames."""
    if not os.path.exists(path):
        raise FileNotFoundError(
            f"{path} not found. Run the simulation first:\n"
            f"  ./build/fastmd data/CG_water_md/fastmd_nvt.conf"
        )

    with open(path, 'rb') as f:
        magic = struct.unpack('i', f.read(4))[0]
        if magic != 0x4d444247:
            raise ValueError(f"Bad magic: {hex(magic)}, expected 0x4d444247")
        natoms = struct.unpack('i', f.read(4))[0]
        ntypes = struct.unpack('i', f.read(4))[0]

    header_size = 12
    frame_size = 8 + 4 + 4 + natoms * 12
    file_size = os.path.getsize(path)
    data_size = file_size - header_size

    if data_size % frame_size != 0:
        raise ValueError(
            f"Trajectory appears incomplete: data_size={data_size}, "
            f"frame_size={frame_size}, remainder={data_size % frame_size}"
        )

    n_frames = data_size // frame_size
    frames = []

    with open(path, 'rb') as f:
        f.seek(header_size)
        for _ in range(n_frames):
            raw = f.read(frame_size)
            step = struct.unpack('q', raw[0:8])[0]
            n = struct.unpack('i', raw[8:12])[0]
            box_L = struct.unpack('f', raw[12:16])[0]
            pos = np.frombuffer(raw[16:16 + n * 12], dtype=np.float32).reshape(n, 3)
            frames.append({'step': step, 'box_L': box_L, 'positions': pos})

    print(f"Loaded {n_frames} frames, natoms={natoms}, ntypes={ntypes}")
    return {'natoms': natoms, 'ntypes': ntypes, 'frames': frames}

traj_path = 'traj.bin'
file_size = os.path.getsize(traj_path) if os.path.exists(traj_path) else 0
print(f"traj.bin: {file_size:,} bytes ({file_size/1024/1024:.1f} MB)")

data = read_trajectory(traj_path)
box_L = data['frames'][0]['box_L']
natoms = data['natoms']
print(f"box_L = {box_L:.4f} nm, r_max = {box_L/2:.4f} nm")


In [ ]:
def compute_rdf(frames, dr, r_max):
    """Compute g(r) from trajectory frames with minimum-image PBC."""
    nbins = int(r_max / dr)
    total_hist = np.zeros(nbins, dtype=np.float64)
    n_frames = len(frames)

    for fi, frame in enumerate(frames):
        pos = frame['positions']
        box_L = frame['box_L']
        n = len(pos)

        for i in range(n):
            delta = pos - pos[i]
            # Minimum image convention
            delta -= box_L * np.round(delta / box_L)
            dist = np.sqrt(np.sum(delta * delta, axis=1))
            # Only upper triangle: j > i
            mask = np.arange(n) > i
            dist = dist[mask]
            # Bin distances
            idx = (dist / dr).astype(np.int32)
            valid = idx < nbins
            np.add.at(total_hist, idx[valid], 1)

        if (fi + 1) % 100 == 0:
            print(f"  processed {fi + 1}/{n_frames} frames")

    # Normalize to g(r)
    r = (np.arange(nbins) + 0.5) * dr
    V = box_L ** 3
    # Shell volume: 4π r² dr
    shell_vol = 4.0 * np.pi * r * r * dr
    # g(r) = total_hist * V / (4πr²dr * N_frames * N*(N-1)/2)
    gr = total_hist * V / (shell_vol * n_frames * natoms * (natoms - 1) / 2.0)

    return r, gr

# Match reference binning: dr=0.003 nm, r_max = box_L/2
dr = 0.003
r_max = box_L / 2.0
print(f"Computing RDF: dr={dr} nm, r_max={r_max:.4f} nm, nbins={int(r_max/dr)}")
print(f"Reference used 501 bins from ~0 to {0.0015 + 500*0.003:.4f} nm")

r_bins, gr = compute_rdf(data['frames'], dr, r_max)
print("RDF computation complete")


In [ ]:
def load_reference_rdf(path):
    """Load reference RDF file (two-column: r(nm) g(r)), skip comments."""
    r_vals, gr_vals = [], []
    with open(path, 'r') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'):
                continue
            parts = line.split()
            if len(parts) >= 2:
                r_vals.append(float(parts[0]))
                gr_vals.append(float(parts[1]))
    return np.array(r_vals), np.array(gr_vals)

ref_path = '1water_oo_rdf.dat'
ref_r, ref_gr = load_reference_rdf(ref_path)
print(f"Loaded reference: {len(ref_r)} points, r in [{ref_r[0]:.4f}, {ref_r[-1]:.4f}] nm")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(ref_r, ref_gr, 'k-', linewidth=1.5, label='LAMMPS reference', alpha=0.8)
ax.plot(r_bins, gr, 'r-', linewidth=1.5, label='fastMD', alpha=0.8)

ax.set_xlabel('r (nm)')
ax.set_ylabel('g(r)')
ax.set_title('CG Water O-O RDF: fastMD vs LAMMPS')
ax.legend()
ax.set_xlim(0, r_max)
ax.grid(True, alpha=0.3)

fig.tight_layout()
fig.savefig('rdf_comparison.png', dpi=150, bbox_inches='tight')
print("Plot saved to rdf_comparison.png")
plt.show()
